# Day 050 — Exercise 3: train_and_evaluate

**What you'll build:** `train_and_evaluate(df, target_col, ...) -> dict` — automatically select numeric features, scale them, train LinearRegression with 5-fold CV, and evaluate on a held-out test set.

**Why it matters:** The Insight Engine needs to work on any tabular dataset without the user specifying features manually. `train_and_evaluate` encodes the full Day 48+49 pipeline — feature selection, scaling, CV, evaluation — in one auto-pilot function.

## Provided: Setup + load_and_clean + run_eda

In [ ]:
import io
import warnings
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import ollama
warnings.filterwarnings('ignore')


def make_sample_data(n: int = 300, seed: int = 42) -> pd.DataFrame:
    """
    Retail sales dataset.
    Columns: date (str), region, category, units_sold, price, discount, revenue.
    Revenue = units_sold*4 + price*1.5 - discount*150 + noise (5% nulls injected).
    """
    rng      = np.random.default_rng(seed)
    dates    = pd.date_range('2023-01-01', periods=n, freq='D').strftime('%Y-%m-%d')
    region   = rng.choice(['North', 'South', 'East', 'West'], n)
    category = rng.choice(['Electronics', 'Clothing', 'Food', 'Books'], n)
    units    = rng.integers(1, 50, n)
    price    = rng.uniform(5.0, 200.0, n).round(2)
    discount = rng.choice([0.0, 0.05, 0.10, 0.15, 0.20], n)
    revenue  = (units * 4.0 + price * 1.5 - discount * 150
                + rng.standard_normal(n) * 20).round(2)
    null_idx = rng.choice(n, size=max(1, int(n * 0.05)), replace=False)
    revenue  = revenue.astype(float)
    revenue[null_idx] = np.nan
    return pd.DataFrame({
        'date':       pd.Series(dates),
        'region':     region,
        'category':   category,
        'units_sold': units,
        'price':      price,
        'discount':   discount,
        'revenue':    revenue,
    })


def load_and_clean(source) -> pd.DataFrame:
    """
    Load from CSV string / file path / DataFrame and clean.

    Steps applied in order:
      1. Parse source into a DataFrame
      2. Detect and parse date/time columns to datetime64
      3. Fill numeric NaN with column median
      4. Drop exact duplicate rows
    """
    if isinstance(source, pd.DataFrame):
        df = source.copy()
    elif isinstance(source, str) and ('\n' in source or ',' in source[:200]):
        df = pd.read_csv(io.StringIO(source))
    else:
        df = pd.read_csv(source)

    # Detect date columns by name
    for col in df.columns:
        if any(kw in col.lower() for kw in ('date', 'time', 'created', 'updated')):
            try:
                df[col] = pd.to_datetime(df[col], errors='coerce')
            except Exception:
                pass

    # Fill numeric NaN with column median
    for col in df.select_dtypes(include='number').columns:
        median = df[col].median()
        df[col] = df[col].fillna(median)

    # Drop duplicates
    df = df.drop_duplicates().reset_index(drop=True)
    return df


def run_eda(df: pd.DataFrame) -> dict:
    """
    Compute an EDA summary dict with keys:
        shape, columns, dtypes, null_counts,
        numeric_summary, correlations, category_counts,
        numeric_cols, cat_cols
    """
    num_cols = df.select_dtypes(include='number').columns.tolist()
    cat_cols = df.select_dtypes(include='object').columns.tolist()

    numeric_summary = {}
    for col in num_cols:
        s = df[col].dropna()
        numeric_summary[col] = {
            'mean':   round(float(s.mean()),   4),
            'std':    round(float(s.std()),    4),
            'min':    round(float(s.min()),    4),
            'max':    round(float(s.max()),    4),
            'median': round(float(s.median()), 4),
        }

    correlations = {}
    if len(num_cols) >= 2:
        cm = df[num_cols].corr()
        for col in num_cols:
            correlations[col] = {
                other: round(float(cm.loc[col, other]), 4)
                for other in num_cols if other != col
            }

    category_counts = {
        col: df[col].value_counts().head(10).to_dict()
        for col in cat_cols
    }

    return {
        'shape':           {'rows': int(df.shape[0]), 'cols': int(df.shape[1])},
        'columns':         df.columns.tolist(),
        'dtypes':          {c: str(t) for c, t in df.dtypes.items()},
        'null_counts':     df.isnull().sum().to_dict(),
        'numeric_summary': numeric_summary,
        'correlations':    correlations,
        'category_counts': category_counts,
        'numeric_cols':    num_cols,
        'cat_cols':        cat_cols,
    }

## Your Implementation

In [ ]:
def train_and_evaluate(df: pd.DataFrame, target_col: str,
                        test_size: float = 0.2,
                        random_state: int = 42) -> dict:
    """
    Auto-select numeric features (exclude target_col), scale with
    StandardScaler, 5-fold cross-validate, fit, and evaluate on test set.

    Returns dict with keys:
        target, features, cv_r2 (mean/std),
        test_r2, test_rmse, test_mae,
        coefficients (feature → value), n_train, n_test
    """
    if target_col not in df.columns:
        return {'error': f'target column {target_col!r} not found'}

    num_cols     = df.select_dtypes(include='number').columns.tolist()
    feature_cols = [c for c in num_cols if c != target_col]

    if not feature_cols:
        return {'error': 'no numeric feature columns found'}

    sub = df[feature_cols + [target_col]].dropna()
    X   = sub[feature_cols]
    y   = sub[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    scaler = StandardScaler()
    X_tr_s = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols)
    X_te_s = pd.DataFrame(scaler.transform(X_test),      columns=feature_cols)

    # TODO: 5-fold CV
    # kf    = KFold(n_splits=5, shuffle=True, random_state=42)
    # model = LinearRegression()
    # cv_scores = cross_val_score(model, X_tr_s, y_train, cv=kf, scoring='r2')

    # TODO: fit + evaluate
    # model.fit(X_tr_s, y_train)
    # y_pred = model.predict(X_te_s)

    # TODO: return {
    #     'target':    target_col,
    #     'features':  feature_cols,
    #     'cv_r2':     {'mean': round(float(cv_scores.mean()), 4),
    #                   'std':  round(float(cv_scores.std()),  4)},
    #     'test_r2':   round(float(r2_score(y_test, y_pred)), 4),
    #     'test_rmse': round(float(np.sqrt(mean_squared_error(y_test, y_pred))), 2),
    #     'test_mae':  round(float(mean_absolute_error(y_test, y_pred)), 2),
    #     'coefficients': {
    #         col: round(float(c), 4)
    #         for col, c in zip(feature_cols, model.coef_)
    #     },
    #     'n_train': int(len(X_train)),
    #     'n_test':  int(len(X_test)),
    # }
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    df = load_and_clean(make_sample_data(200))

    # Check 1: returns dict
    try:
        result = train_and_evaluate(df, 'revenue')
        assert isinstance(result, dict), \
            f'expected dict, got {type(result).__name__}'
        assert 'error' not in result, f"error: {result.get('error')}"
        passed += 1; print(f'\u2705 Check 1: train_and_evaluate returns dict')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: all required keys present
    try:
        for k in ('target', 'features', 'cv_r2', 'test_r2',
                  'test_rmse', 'test_mae', 'coefficients', 'n_train', 'n_test'):
            assert k in result, f'missing key: {k!r}'
        passed += 1; print(f'\u2705 Check 2: all 9 required keys present')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: test_r2 > 0.5 (model learns from data)
    try:
        r2 = result['test_r2']
        assert r2 > 0.5, \
            f'test_r2 should be > 0.5 on this dataset, got {r2}'
        passed += 1; print(f'\u2705 Check 3: test_r2={r2} > 0.5')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: coefficients has one entry per feature
    try:
        coef = result['coefficients']
        feats = result['features']
        assert len(coef) == len(feats), \
            f'coefficients has {len(coef)} entries but {len(feats)} features'
        for f in feats:
            assert f in coef, f'missing coefficient for {f!r}'
        passed += 1; print(f'\u2705 Check 4: coefficients has {len(coef)} entries matching features')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: n_train + n_test == total rows used
    try:
        n_used = result['n_train'] + result['n_test']
        assert n_used <= len(df), \
            f'n_train + n_test = {n_used} exceeds df length {len(df)}'
        assert result['n_test'] > 0 and result['n_train'] > 0
        passed += 1; print(f"\u2705 Check 5: n_train={result['n_train']}, n_test={result['n_test']}")
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def train_and_evaluate(df: pd.DataFrame, target_col: str,
                        test_size: float = 0.2,
                        random_state: int = 42) -> dict:
    """
    Auto-select numeric features, scale, train LinearRegression with 5-fold CV,
    and evaluate on a held-out test set.

    Returns dict with keys:
        target, features, cv_r2 (mean/std), test_r2, test_rmse, test_mae,
        coefficients (feature → value), n_train, n_test
    """
    if target_col not in df.columns:
        return {'error': f'target column {target_col!r} not found'}

    num_cols     = df.select_dtypes(include='number').columns.tolist()
    feature_cols = [c for c in num_cols if c != target_col]

    if not feature_cols:
        return {'error': 'no numeric feature columns found'}

    sub   = df[feature_cols + [target_col]].dropna()
    X     = sub[feature_cols]
    y     = sub[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    scaler   = StandardScaler()
    X_tr_s   = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols)
    X_te_s   = pd.DataFrame(scaler.transform(X_test),      columns=feature_cols)

    model = LinearRegression()
    kf    = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_tr_s, y_train, cv=kf, scoring='r2')

    model.fit(X_tr_s, y_train)
    y_pred = model.predict(X_te_s)

    return {
        'target':    target_col,
        'features':  feature_cols,
        'cv_r2':     {'mean': round(float(cv_scores.mean()), 4),
                      'std':  round(float(cv_scores.std()),  4)},
        'test_r2':   round(float(r2_score(y_test, y_pred)),                         4),
        'test_rmse': round(float(np.sqrt(mean_squared_error(y_test, y_pred))),       2),
        'test_mae':  round(float(mean_absolute_error(y_test, y_pred)),               2),
        'coefficients': {
            col: round(float(c), 4)
            for col, c in zip(feature_cols, model.coef_)
        },
        'n_train': int(len(X_train)),
        'n_test':  int(len(X_test)),
    }
```

</details>